To build a complete RAG system completely from scratch without any pre-existing model weights (no OpenAI, no Llama, no Qwen, no Hugging Face weights), you must build and train a custom neural architecture using your local JSON data.Because you want this model to actually generate text answers (instead of just classifying or grabbing a text chunk), a standard CNN or simple classification network will not work. You must build a sequence-to-sequence Transformer Encoder-Decoder or a Causal Autoregressive Decoder from scratch using a deep learning library like PyTorch.

Architectural Reality Check for a From-Scratch TransformerWhen you build a text generator from scratch, the model starts completely blank. It does not even know English. It has to learn syntax, grammar, spelling, and semantic alignment entirely from your knowledge_base.json and evaluation_set.json files.Because your custom files are relatively small compared to enterprise datasets (like Wikipedia or Common Crawl), a large model would instantly overfit and spit out gibberish. Therefore, we must design a highly compact Mini-Transformer Architecture explicitly sized to learn from your localized JSON structural payload.

The Complete "From-Scratch" Transformer RAG ScriptSave this file as train_scratch_transformer_rag.py. This script handles the raw JSON structures, builds a custom character-level tokenizer, sets up multi-head self-attention layers, optimizes cross-entropy token loss parameters, trains the weights, and runs real-time user inference.

1st Iteration - Did Not Run


2nd Iteration - Did Run

3rd Iteration - Did Run 
The generated output was "Generated Response: "the the a a the saha the a the the the the the saha saha a the a the the a saha the a saha the the a a a"


4th Iteration - Did Run, but the Generated Language Response: "the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the"


***Code***

**Cell 1** -Environment Installation and Setup

In [13]:
# # Run this cell to ensure you have the raw processing packages installed
# !pip install torch pypdf

Test to know if the terminal installed line- "pip install torch pypdf" has its packages installed in the virtual environment or not.

In [1]:
try:
    import torch
    import pypdf
    print(f"✅ Success! torch version: {torch.__version__}")
    print(f"✅ Success! pypdf version: {pypdf.__version__}")
except ImportError as e:
    print(f"❌ Error: {e}")

✅ Success! torch version: 2.11.0+cu128
✅ Success! pypdf version: 6.14.2


In [2]:
import torch
torch.cuda.is_available()


True

### *Cell 2: Ingestion & Mock JSON Data Set Generation*

Old Cell 2

In [ ]:
# import os
# import json

# # Define our target data file paths
# KB_FILE = "knowledge_base1.json"
# EVAL_FILE = "evaluation_set1.json"

# # Automate generating structural sample files if they do not exist on your drive
# if not os.path.exists(KB_FILE):
#     kb_mock = [
#         {"id": 1, "text": "Viksit Bharat is a development blueprint for economic expansion by 2047."},
#         {"id": 2, "text": "Surakshit Bharat focuses on robust cybersecurity and defense perimeter hardening."},
#         {"id": 3, "text": "The technology updates introduces local AI frameworks running on consumer hardware configurations."}
#     ]
#     with open(KB_FILE, "w") as f:
#         json.dump(kb_mock, f, indent=4)
#     print(f"Generated sample file: {KB_FILE}")

# if not os.path.exists(EVAL_FILE):
#     eval_mock = [
#         {"query": "What is Viksit Bharat blueprint?", "expected_answer": "Blueprint for economic expansion by 2047."},
#         {"query": "What does Surakshit Bharat focus on?", "expected_answer": "Focuses on robust cybersecurity and defense perimeter hardening."},
#         {"query": "What does the technology updates introduce?", "expected_answer": "Introduces local AI frameworks running on consumer hardware."}
#     ]
#     with open(EVAL_FILE, "w") as f:
#         json.dump(eval_mock, f, indent=4)
#     print(f"Generated sample file: {EVAL_FILE}")

# # Load raw JSON payloads directly into working memory arrays
# with open(KB_FILE, "r") as f:
#     kb_data = json.load(f)
# with open(EVAL_FILE, "r") as f:
#     eval_data = json.load(f)

# print(f"\nSuccessfully loaded {len(kb_data)} knowledge blocks and {len(eval_data)} evaluation test sets.")



Generated sample file: knowledge_base1.json
Generated sample file: evaluation_set1.json

Successfully loaded 3 knowledge blocks and 3 evaluation test sets.


**Updated Cell 2**

In [3]:
import os
import json

# Your exact local file paths
KB_FILE = "knowledge_base.json"
EVAL_FILE = "evaluation_set.json"

# Check if the files exist before reading
if not os.path.exists(KB_FILE) or not os.path.exists(EVAL_FILE):
    raise FileNotFoundError(
        f"Ensure '{KB_FILE}' and '{EVAL_FILE}' containing your Venezuela/CBI "
        f"data are placed in this exact folder directory: {os.getcwd()}"
    )

# Load your actual data assets
with open(KB_FILE, "r", encoding="utf-8") as f:
    kb_data = json.load(f)
with open(EVAL_FILE, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("==== DATA ACQUISITION SUCCESS ====")
print(f"Loaded {len(kb_data)} knowledge chunks (Fields parsed: 'chunk_id', 'text', 'metadata')")
print(f"Loaded {len(eval_data)} evaluation items (Fields parsed: 'user_query', 'ground_truth_answer')")


==== DATA ACQUISITION SUCCESS ====
Loaded 6 knowledge chunks (Fields parsed: 'chunk_id', 'text', 'metadata')
Loaded 5 evaluation items (Fields parsed: 'user_query', 'ground_truth_answer')


### *Cell 3: Custom Character-Level Tokenizer & Vocabulary Matrix*

Old Cell 3

In [ ]:
# # Gather every distinct character across our textual corpus to build an alphabet map
# raw_corpus = ""
# for item in kb_data: raw_corpus += item["text"] + " "
# for item in eval_data: raw_corpus += item["query"] + " " + item["expected_answer"] + " "

# # Define special boundary control tracking tokens
# chars = sorted(list(set(raw_corpus)))
# vocab = ["<pad>", "<sos>", "<eos>"] + chars

# # Construct bi-directional index dictionaries
# char_to_id = {ch: i for i, ch in enumerate(vocab)}
# id_to_char = {i: ch for i, ch in enumerate(vocab)}

# VOCAB_SIZE = len(vocab)
# PAD_IDX = char_to_id["<pad>"]
# SOS_IDX = char_to_id["<sos>"]
# EOS_IDX = char_to_id["<eos>"]

# print(f"Unique Characters Found: {len(chars)}")
# print(f"Total Model Vocabulary Size (Alphabet + Special Tokens): {VOCAB_SIZE}")
# print(f"Control Mappings -> SOS: {SOS_IDX}, EOS: {EOS_IDX}, PAD: {PAD_IDX}")

# def encode_text(text, max_len=128):
#     """Translates raw string characters into a list of vocabulary numerical IDs."""
#     tokens = [SOS_IDX] + [char_to_id[c] for c in text if c in char_to_id] + [EOS_IDX]
#     if len(tokens) < max_len:
#         tokens += [PAD_IDX] * (max_len - len(tokens))
#     return tokens[:max_len]

# # Let's inspect a quick transformation test
# sample_sentence = "Viksit Bharat"
# sample_encoded = encode_text(sample_sentence, max_len=20)
# print(f"\nTest String Encoding: '{sample_sentence}' -> {sample_encoded}")


KeyError: 'query'

**Cell 3** : 2nd Iteration

In [ ]:
# # Gather character distributions across your actual JSON dictionary keys
# raw_corpus = ""

# # Map to your actual key name: item["text"]
# for item in kb_data: 
#     raw_corpus += item["text"] + " "

# # Map to your actual key names: item["user_query"] and item["ground_truth_answer"]
# for item in eval_data: 
#     raw_corpus += item["user_query"] + " " + item["ground_truth_answer"] + " "

# # Define special structural control tokens
# chars = sorted(list(set(raw_corpus)))
# vocab = ["<pad>", "<sos>", "<eos>"] + chars

# # Construct bidirectional conversion dictionaries
# char_to_id = {ch: i for i, ch in enumerate(vocab)}
# id_to_char = {i: ch for i, ch in enumerate(vocab)}

# VOCAB_SIZE = len(vocab)
# PAD_IDX = char_to_id["<pad>"]
# SOS_IDX = char_to_id["<sos>"]
# EOS_IDX = char_to_id["<eos>"]

# print(f"Unique Characters Found in Your Corpus: {len(chars)}")
# print(f"Total Model Vocabulary Matrix Size   : {VOCAB_SIZE}")

# def encode_text(text, max_len=128):
#     """Translates strings into a list of vocabulary numerical IDs."""
#     tokens = [SOS_IDX] + [char_to_id[c] for c in text if c in char_to_id] + [EOS_IDX]
#     if len(tokens) < max_len:
#         tokens += [PAD_IDX] * (max_len - len(tokens))
#     return tokens[:max_len]

# # Quick validation sanity check with your actual text topic
# test_string = "Operation Chakra-VI"
# print(f"\nVerification Test: '{test_string}' -> {encode_text(test_string, max_len=25)}")


Unique Characters Found in Your Corpus: 65
Total Model Vocabulary Matrix Size   : 68

Verification Test: 'Operation Chakra-VI' -> [1, 33, 56, 45, 58, 41, 60, 49, 55, 54, 3, 22, 48, 41, 51, 58, 41, 8, 39, 28, 2, 0, 0, 0, 0]


**Updated Cell 3** : 3rd Iteration

In [4]:
# Upgraded Cell 3: Word-Level Vocabulary Engine

# Compile every distinct word across your real text files to build a word-index dictionary
raw_corpus = ""
for item in kb_data: raw_corpus += item["text"] + " "
for item in eval_data: raw_corpus += item["user_query"] + " " + item["ground_truth_answer"] + " "

# Clean basic punctuation to extract raw words
processed_words = []
for word in raw_corpus.lower().split():
    processed_words.append(word.strip(".,;:!?()\""))

# Remove duplicates and register special token flags
unique_words = sorted(list(set(processed_words)))
vocab = ["<pad>", "<sos>", "<eos>", "<unk>"] + unique_words

# Construct word conversion mappings
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
PAD_IDX = word_to_id["<pad>"]
SOS_IDX = word_to_id["<sos>"]
EOS_IDX = word_to_id["<eos>"]
UNK_IDX = word_to_id["<unk>"]

print(f"Total Unique Vocabulary Words Found: {len(unique_words)}")
print(f"Total Matrix Input Size (Vocab + Controls): {VOCAB_SIZE}")

def encode_text(text, max_len=128):
    """Translates strings into a list of word vocabulary numerical IDs."""
    tokens = [SOS_IDX]
    for word in text.lower().split():
        word = word.strip(".,;:!?()\"")
        tokens.append(word_to_id.get(word, UNK_IDX)) # Fallback to <unk> if word is missing
    tokens.append(EOS_IDX)
    
    if len(tokens) < max_len:
        tokens += [PAD_IDX] * (max_len - len(tokens))
    return tokens[:max_len]

# Run a sample validation slice
print(f"\nVerification Test: 'Operation Chakra-VI' -> {encode_text('Operation Chakra-VI', max_len=6)}")


Total Unique Vocabulary Words Found: 226
Total Matrix Input Size (Vocab + Controls): 230

Verification Test: 'Operation Chakra-VI' -> [1, 152, 47, 2, 0, 0]


### *Cell 4: Native Jaccard Text Overlap Search Engine (The Retriever Node)*

**Updated Cell 4**

In [5]:
def manual_jaccard_retriever(query, kb_list):
    """A zero-dependency search engine tracking maximum word intersection metrics."""
    q_words = set(query.lower().split())
    best_chunk = kb_list[0]["text"] if len(kb_list) > 0 else ""
    best_score = -1
    
    for item in kb_list:
        # Maps natively to your document text data stream layer
        chunk_words = set(item["text"].lower().split())
        intersection = q_words.intersection(chunk_words)
        union = q_words.union(chunk_words)
        score = len(intersection) / len(union) if len(union) > 0 else 0
        
        if score > best_score:
            best_score = score
            best_chunk = item["text"]
            
    return best_chunk

# Test-run the retriever with one of your actual evaluation query types
sample_query = "What is the name of the CBI operation against digital arrest scams?"
matched_context = manual_jaccard_retriever(sample_query, kb_data)

print(f"Query Test      : {sample_query}")
print(f"Retrieved Block : {matched_context[:120]}...")


Query Test      : What is the name of the CBI operation against digital arrest scams?
Retrieved Block : As part of Operation Chakra-VI, the CBI arrested two key accused: Naresh from Chennai, Tamil Nadu, and Sanjib Saha from ...


### *Cell 5: Tensor Dataset Mapper and PyTorch DataLoader*

**Updated Cell 5**

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class RAGDataset(Dataset):
    """Maps custom text formats to coupled, padded training tensor arrays."""
    def __init__(self, eval_set, kb_set, max_len=128):
        self.samples = []
        for item in eval_set:
            # 1. Access your real query key string: item["user_query"]
            query = item["user_query"]
            
            # 2. Execute local retrieval search pass
            context = manual_jaccard_retriever(query, kb_set)
            
            # 3. Stitch prompt structures into a cohesive sequence matrix layout
            input_text = f"Context: {context} | Query: {query} -> Answer: "
            
            # 4. Access your real output key target string: item["ground_truth_answer"]
            target_text = item["ground_truth_answer"]
            
            self.samples.append({
                "input_ids": torch.tensor(encode_text(input_text, max_len), dtype=torch.long),
                "target_ids": torch.tensor(encode_text(target_text, max_len), dtype=torch.long)
            })
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

# Instantiate dataset and batch loaders using your actual variables
dataset = RAGDataset(eval_data, kb_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Successfully constructed PyTorch DataLoader around your custom JSON keys.")
print(f"Total training data vectors loaded into grid: {len(dataset)}")


Successfully constructed PyTorch DataLoader around your custom JSON keys.
Total training data vectors loaded into grid: 5


### *Cell 6: Building the Transformer Model Architecture from Scratch*

**Cell 6**

In [ ]:
# import torch.nn as nn

# class MiniCausalTransformer(nn.Module):
#     """
#     A pure PyTorch Causal Language Transformer Model designed from scratch.
#     Contains character token lookups, spatial positions, and self-attention blocks.
#     """
#     def __init__(self, vocab_size, embedding_dim=64, num_heads=4, hidden_dim=128, max_len=128):
#         super(MiniCausalTransformer, self).__init__()
#         # 1. Token Embedding Matrix Lookup Array
#         self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
#         # 2. Spatial Index Tracking Position Layer
#         self.position_embedding = nn.Embedding(max_len, embedding_dim)
        
#         # 3. Core Multi-Head Self-Attention Assembly Layer Blocks
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=embedding_dim,
#             nhead=num_heads,
#             dim_feedforward=hidden_dim,
#             batch_first=True,
#             norm_first=True
#         )
#         self.transformer_block = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
#         # 4. Dense Density Network: Projects matrix weights back to vocabulary character logits
#         self.fc_out = nn.Linear(embedding_dim, vocab_size)
        
#     def forward(self, x):
#         batch_size, seq_len = x.shape
#         positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
        
#         # Merge Token representations and Spatial Position coordinate structures together
#         out = self.token_embedding(x) + self.position_embedding(positions)
        
#         # Generate causal masking boundaries to block looking at upcoming characters
#         mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        
#         out = self.transformer_block(out, mask=mask, is_causal=True)
#         return self.fc_out(out)

# # Mount the blank model grid directly onto your available processing hardware hardware
# model = MiniCausalTransformer(VOCAB_SIZE)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# print(f"Model mounted cleanly on compute layer: {device}")


Model mounted cleanly on compute layer: cuda


C:\Users\lanst\AppData\Local\Temp\ipykernel_23756\2927587473.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_block = nn.TransformerEncoder(encoder_layer, num_layers=2)


**Updated Cell 6** - 4th Iteration

In [7]:
# Updated Cell 6: 4th Iteration

import torch.nn as nn

class MiniCausalTransformer(nn.Module):
    """
    A pure PyTorch Causal Language Transformer Model designed from scratch.
    Expanded dimensions to force memorisation on localized data profiles.
    """
    def __init__(self, vocab_size, embedding_dim=128, num_heads=4, hidden_dim=256, max_len=128):
        super(MiniCausalTransformer, self).__init__()
        # 1. Token Embedding Matrix Lookup Array
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        # 2. Spatial Index Tracking Position Layer
        self.position_embedding = nn.Embedding(max_len, embedding_dim)
        
        # 3. Structural Multi-Head Self-Attention Core (4 Layers Deep)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True,
            norm_first=True
        )
        self.transformer_block = nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        # 4. Projection Layer: Maps multi-dimensional outputs back to word vocabulary sizes
        self.fc_out = nn.Linear(embedding_dim, vocab_size)
        
    def forward(self, x):
        batch_size, seq_len = x.shape
        positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
        
        # Merge semantic token arrays with spatial position tracking layers
        out = self.token_embedding(x) + self.position_embedding(positions)
        
        # Generate a lower-triangular causal mask to block future token sightlines
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        
        out = self.transformer_block(out, mask=mask, is_causal=True)
        return self.fc_out(out)

# =====================================================================
# MODEL INITIALIZATION & HARDWARE COUPLING
# =====================================================================
# Set high-capacity hyper-parameters to scale model parameters
EMBEDDING_DIM = 128
NUM_HEADS = 4
HIDDEN_DIM = 256

# Instantiate the custom network layers
model = MiniCausalTransformer(
    vocab_size=VOCAB_SIZE, 
    embedding_dim=EMBEDDING_DIM, 
    num_heads=NUM_HEADS, 
    hidden_dim=HIDDEN_DIM,
    max_len=128
)

# Detect if your laptop graphics card driver setup completed successfully
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model mounted cleanly on compute layer: {device}")


Model mounted cleanly on compute layer: cuda


C:\Users\lanst\AppData\Local\Temp\ipykernel_3852\995997839.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_block = nn.TransformerEncoder(encoder_layer, num_layers=4)


In [8]:
import torch
print("Is GPU available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Card Name   :", torch.cuda.get_device_name(0))


Is GPU available?: True
GPU Card Name   : NVIDIA GeForce RTX 5050 Laptop GPU


### *Figuring out the Correct Pytorch Version to be Compatible with RTX 5050 GPU.*

In [ ]:
# !pip uninstall -y torch
# !pip install torch --index-url 

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1


You can safely remove it manually.

[optparse.groups]Usage:[/]   
  pip install \[options] <requirement specifier> \[package-index-options] ...
  pip install \[options] -r <requirements file> \[package-index-options] ...
  pip install \[options] [-e] <vcs project url> ...
  pip install \[options] [-e] <local project path> ...
  pip install \[options] <archive url/path> ...

--index-url option requires 1 argument


In [ ]:
# !pip install torch --index-url https://pytorch.org
# !pip install torch --index-url https://pytorch.org

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB 16.8 MB/s eta 0:02:36
   ---------------------------------------- 0.0/2.6 GB 11.6 MB/s eta 0:03:47
   ---------------------------------------- 0.0/2.6 GB 9.8 MB/s eta 0:04:28
   ---------------------------------------- 0.0/2.6 GB 8.3 MB/s eta 0:05:14
   ---------------------------------------- 0.0/2.6 GB 7.6 MB/s eta 0:05:42
   ---------------------------------------- 0.0/2.6 GB 7.2 MB/s eta 0:06:05
   ---------------------------------------- 0.0/2.6 GB 6.8 MB/s eta 0:06:25
   ---------------------------------------- 0.0/2.6 GB 6.5 MB/s eta 0:06:41
   ---------------------------------------- 0.0/2.6 GB 6.4 MB/s eta 0:06:51
   ---------------------------------------- 0.0/2.6 GB 6.2 MB/s eta 0:07:02
   ---------------------------------------- 0.0/2.6 GB 5.8 MB/s eta 0:07:28
   -------------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# # 1. Cleanly uninstall the existing PyTorch, TorchVision, and TorchAudio packages
# !pip uninstall -y torch torchvision torchaudio

# # 2. Install the new version targeting the CUDA 13.0 warehouse index
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130


Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torch-2.12.1%2Bcu130-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached https://download-r2.pytorch.org/whl/cu130/torchvision-0.27.1%2Bcu130-cp313-cp313-win_amd64.whl.metadata (5.6 kB)
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB 583.9 kB/s eta 0:54:59
   ---------------------------------------- 0.0/1.9 GB 583.9 kB/s eta 0:54:59
   ---------------------------------------- 0.0/1.9 GB 617.1 kB/s eta 0:52:01
   ---------------------------------------- 0.0/1.9 GB 617.1 kB/s eta 0:52:01
   ---------------------------------------- 0.0/1.9 GB 614.6 kB/s eta 0:52:13
   -----------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### *Finally figured out the correct Pytorch version to be used in this Program.* !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# # 1. Clean out the broken CUDA 13 components entirely
# !pip uninstall -y torch torchvision torchaudio

# # 2. Force-install the compatible CUDA 12.8 wheel distributions
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128


Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-win_amd64.whl.metadata (5.6 kB)
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 1.7 MB/s eta 0:27:03
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:28:17
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:28:44
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:29:13
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:29:14
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:29:13
   ---------------------------------------- 0.0/2.8 GB 1.6 MB/s eta 0:29:07
   -------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### *Cell 7: Neural Convergence and Weight Training Parameters Loop*

Cell 7

In [ ]:
# import torch.optim as optim

# # Categorical Cross Entropy handles multi-character token prediction probabilities
# criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
# # Adam optimizer scales step matrices based on standard vector loss curves
# optimizer = optim.Adam(model.parameters(), lr=0.005)

# print("Starting custom model optimization training loops...")
# epochs = 300 # Run optimization backpropagation calculations 300 times

# model.train()
# for epoch in range(epochs):
#     epoch_loss = 0
#     for batch in dataloader:
#         inputs = batch["input_ids"].to(device)
#         targets = batch["target_ids"].to(device)
        
#         # Clear out historic parameter gradient updates tracking frames
#         optimizer.zero_grad()
        
#         # Forward pass: Run raw multiplication tensors across active layer matrices
#         logits = model(inputs)
        
#         # Re-shape values to match traditional categorical loss formats
#         loss = criterion(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        
#         # Backpropagation: Calculate calculus error vectors (gradients)
#         loss.backward()
#         # Calibration Step: Readjust weights across the neural connections
#         optimizer.step()
        
#         epoch_loss += loss.item()
        
#     if (epoch + 1) % 50 == 0:
#         print(f"Epoch [{epoch+1}/{epochs}] | Parameter Error Rate (Loss Value): {epoch_loss:.4f}")

# print("\nModel Training Cycle Completed successfully. Internal weights locked.")


Starting custom model optimization training loops...
Epoch [50/300] | Parameter Error Rate (Loss Value): 1.7349
Epoch [100/300] | Parameter Error Rate (Loss Value): 1.7915
Epoch [150/300] | Parameter Error Rate (Loss Value): 1.1434
Epoch [200/300] | Parameter Error Rate (Loss Value): 1.0888
Epoch [250/300] | Parameter Error Rate (Loss Value): 1.5758
Epoch [300/300] | Parameter Error Rate (Loss Value): 1.1497

Model Training Cycle Completed successfully. Internal weights locked.


**Updated Cell 7** : 3rd Iteration, 4th Iteration

In [9]:
import torch.optim as optim

# Categorical Cross Entropy handles multi-word prediction paths
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
# Adam optimizer scales step matrices
# Iteration 4
optimizer = optim.Adam(model.parameters(), lr=0.001) 



print("Starting custom model optimization training loops...")
# epochs = 300 # Run optimization 300 times to converge weights
# Iteration 4
epochs = 1000 # Run optimization 1000 times to converge weights

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for batch in dataloader:
        inputs = batch["input_ids"].to(device)
        targets = batch["target_ids"].to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        
        loss = criterion(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Parameter Error Rate (Loss): {epoch_loss:.4f}")

print("\nModel weights successfully calibrated and locked.")


Starting custom model optimization training loops...
Epoch [50/1000] | Parameter Error Rate (Loss): 1.1271
Epoch [100/1000] | Parameter Error Rate (Loss): 1.7192
Epoch [150/1000] | Parameter Error Rate (Loss): 0.8282
Epoch [200/1000] | Parameter Error Rate (Loss): 0.6638
Epoch [250/1000] | Parameter Error Rate (Loss): 0.7711
Epoch [300/1000] | Parameter Error Rate (Loss): 0.7023
Epoch [350/1000] | Parameter Error Rate (Loss): 0.8305
Epoch [400/1000] | Parameter Error Rate (Loss): 0.9739
Epoch [450/1000] | Parameter Error Rate (Loss): 0.9287
Epoch [500/1000] | Parameter Error Rate (Loss): 0.6858
Epoch [550/1000] | Parameter Error Rate (Loss): 0.7125
Epoch [600/1000] | Parameter Error Rate (Loss): 0.6898
Epoch [650/1000] | Parameter Error Rate (Loss): 1.0942
Epoch [700/1000] | Parameter Error Rate (Loss): 0.9859
Epoch [750/1000] | Parameter Error Rate (Loss): 0.8512
Epoch [800/1000] | Parameter Error Rate (Loss): 0.6369
Epoch [850/1000] | Parameter Error Rate (Loss): 0.7135
Epoch [900/10

### *Cell 8: Real-Time User Ingestion Inference Query (The RAG Chat Runtime)*

Cell 8

In [ ]:
# model.eval()

# # Input whatever test question you want to query against your newly calibrated model
# user_query = "What are the locations of the accused with respect to the digital arrests conducted by the CBI?"

# # Step A: Run Jaccard Search against your database entries
# retrieved_context = manual_jaccard_retriever(user_query, kb_data)

# # Step B: Reassemble identical structural formatting used during training loops
# inference_input = f"Context: {retrieved_context} | Query: {user_query} -> Answer: "
# input_tokens = encode_text(inference_input)
# input_tensor = torch.tensor([input_tokens], dtype=torch.long).to(device)

# generated_chars = ""
# print(f"Evaluating Query Sequence: '{user_query}'")

# # Step C: Token-by-token character prediction processing loop
# with torch.no_grad():
#     for _ in range(60): # Hard constraints to prevent infinite loops if model goes unstable
#         logits = model(input_tensor)
#         # Pull classification distribution logs for the final character slot position
#         next_token_logits = logits[0, -1, :]
#         # Target the vocabulary character ID with the maximum numerical score
#         predicted_token_id = torch.argmax(next_token_logits).item()
        
#         # End sequence generation if the model throws the End-of-String tag
#         if predicted_token_id == EOS_IDX:
#             break
            
#         # Translate the numerical ID back into an alphanumeric symbol string
#         char = id_to_char.get(predicted_token_id, "")
#         generated_chars += char
        
#         # Concat the newly found token back into our active input list tensor
#         new_token_tensor = torch.tensor([[predicted_token_id]], dtype=torch.long).to(device)
#         input_tensor = torch.cat((input_tensor, new_token_tensor), dim=1)[:, :128]

# print("\n================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================")
# print(f"Retrieved Document Block Context: \"{retrieved_context}\"")
# print(f"Generated Language Model Response: \"{generated_chars.strip()}\"")
# print("=====================================================================")


Evaluating Query Sequence: 'What are the locations of the accused with respect to the digital arrests conducted by the CBI?'

================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================
Retrieved Document Block Context: "As part of Operation Chakra-VI, the CBI arrested two key accused: Naresh from Chennai, Tamil Nadu, and Sanjib Saha from Kolkata, West Bengal. The accused were involved in incorporating shell companies and managing mule bank accounts. These fraudulent bank accounts were utilized to launder approximately ₹2 crore of suspected proceeds generated from the digital arrest scams."
Generated Language Model Response: "ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss"


**Cell 8** : 3rd Iteration

In [ ]:
# # Upgraded Cell 8: Real-Time Inference Interface with Sampling Layers

# import torch.nn.functional as F
# model.eval()

# # Input your target validation question
# user_query = "What are the locations of the accused with respect to the digital arrests conducted by the CBI?"

# # 1. Context Retrieval Search Pass
# retrieved_context = manual_jaccard_retriever(user_query, kb_data)

# # 2. Re-create the training format blueprint string payload
# inference_input = f"context: {retrieved_context} | query: {user_query} -> answer: "
# input_tokens = encode_text(inference_input)
# input_tensor = torch.tensor([input_tokens], dtype=torch.long).to(device)

# generated_words = []

# # 3. Upgraded Word Generation Loop
# with torch.no_grad():
#     for _ in range(30): # Generate up to 30 words max
#         logits = model(input_tensor)
#         next_token_logits = logits[0, -1, :]
        
#         # Apply Temperature Scaling to break repetition patterns
#         temperature = 0.4
#         scaled_logits = next_token_logits / temperature
        
#         # Convert raw logit vectors into clear probability percentages
#         probabilities = F.softmax(scaled_logits, dim=-1)
        
#         # Sample from the top most likely words instead of rigidly picking the absolute highest value
#         predicted_token_id = torch.multinomial(probabilities, num_samples=1).item()
        
#         if predicted_token_id == EOS_IDX:
#             break
            
#         # Translate token numerical index back into a clean string word
#         word = id_to_word.get(predicted_token_id, "")
#         if word not in ["<pad>", "<sos>", "<unk>"]:
#             generated_words.append(word)
        
#         # Concat word matrix token back into your active tensor track list
#         new_token_tensor = torch.tensor([[predicted_token_id]], dtype=torch.long).to(device)
#         input_tensor = torch.cat((input_tensor, new_token_tensor), dim=1)[:, :128]

# print("\n================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================")
# print(f"Retrieved Context: \"{retrieved_context[:100]}...\"")
# print(f"Generated Response: \"{' '.join(generated_words)}\"")
# print("=====================================================================")



================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================
Retrieved Context: "As part of Operation Chakra-VI, the CBI arrested two key accused: Naresh from Chennai, Tamil Nadu, a..."
Generated Response: "the the a a the saha the a the the the the the saha saha a the a the the a saha the a saha the the a a a"


**Updated Cell 8** : 4th Iteration

In [10]:
# Iteration 4

import torch
import torch.nn.functional as F

# Put the model into evaluation mode (disables dropout layers)
model.eval()

# Input whatever validation question you want to query against your model
user_question = "What are the locations of the accused with respect to the digital arrests conducted by the CBI?"

print(f"Evaluating User Query Sequence: '{user_question}'")
print("Searching custom index vectors...")

# 1. LOCAL RETRIEVAL: Search your JSON knowledge base for the most relevant data block
retrieved_context = manual_jaccard_retriever(user_question, kb_data)

# 2.BluePRINT STRUCTURING: Reassemble the exact text layout template used during training
inference_input = f"context: {retrieved_context} | query: {user_question} -> answer: "
input_tokens = encode_text(inference_input)

# Convert the word index list into a PyTorch Tensor matrix and push it to your laptop GPU
input_tensor = torch.tensor([input_tokens], dtype=torch.long).to(device)

generated_words = []

# 3. AUTOREGRESSIVE GENERATION LOOP: Predict next tokens step-by-step
with torch.no_grad():
    for _ in range(40): # Cap generation output at 40 words max
        # Run forward pass matrix math across your transformer layers
        logits = model(input_tensor)
        
        # Read the raw output scores for the very last word position in the sequence
        next_token_logits = logits[0, -1, :]
        
        # RIGID ARGMAX DECODING: Pick the absolute highest scoring word in the vocabulary matrix
        # This completely disables randomness to enforce deterministic, factual replication
        predicted_token_id = torch.argmax(next_token_logits).item()
        
        # Terminate generation immediately if the model throws the End-of-String token
        if predicted_token_id == EOS_IDX:
            break
            
        # Translate the numerical ID back into a clean string word
        word = id_to_word.get(predicted_token_id, "")
        
        # Prevent training boundary tags from leaking into your final answer string
        if word not in ["<pad>", "<sos>", "<unk>"]:
            generated_words.append(word)
            
        # Concat the newest token ID back into your active tensor input line
        # This dynamically updates your sliding context window for the next iteration run
        new_token_tensor = torch.tensor([[predicted_token_id]], dtype=torch.long).to(device)
        input_tensor = torch.cat((input_tensor, new_token_tensor), dim=1)[:, :128]

# Clean up final string list spacing layout adjustments
final_answer = " ".join(generated_words)

print("\n================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================")
print(f"Retrieved Document Context: \"{retrieved_context[:120]}...\"")
print(f"Generated Language Response: \"{final_answer}\"")
print("=====================================================================")


Evaluating User Query Sequence: 'What are the locations of the accused with respect to the digital arrests conducted by the CBI?'
Searching custom index vectors...

================== CUSTOM SYSTEM RAG SEARCH OUTPUT ==================
Retrieved Document Context: "As part of Operation Chakra-VI, the CBI arrested two key accused: Naresh from Chennai, Tamil Nadu, and Sanjib Saha from ..."
Generated Language Response: "the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the"


### *Review*

## 2nd Iteration ##

The reason your model is generating ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss is a classic deep learning bug known as a token collapse or repetition loop.

Because your neural network architecture was initialized from scratch with random numbers and trained on a tiny dataset (only 3 rows), its mathematical weights haven't learned how text generation works yet. During training, the model took a shortcut to lower its error score: it discovered that repeating the most common character (s) over and over was the easiest way to satisfy the loss function math.

To fix this repetition loop and make your from-scratch Transformer generate actual, human-readable answers, we need to apply three critical architectural adjustments.

*Step 1: Replace Character-Level with Word-Level Tokenization*

A character-level model has to learn how to spell words letter-by-letter before it can even attempt to answer a question. This is incredibly difficult for a blank model.
By upgrading to a Word-Level Vocabulary Engine, the model processes entire words (like "Chennai", "Kolkata", or "CBI") as a single unit, eliminating character repetition completely.

*Step 2: Implement Token Temperature Scaling*

In your text generation cell, the code picks the next word using a greedy argmax strategy: torch.argmax(next_token_logits). If the network assigns a microscopic statistical advantage to the word or character s, the model gets locked into an infinite loop of picking s forever.We must introduce a Temperature scaling threshold and a top-k sampling layer to disrupt this tracking pattern and allow the model to explore logical sentence paths.

*🚀 The Complete Updated Notebook Implementation*

Replace your previous notebook cells from Cell 3 onwards with the upgraded code blocks below. This implementation introduces word-level slicing and custom generation sampling layer logic.

*Upgraded Cell 3: Word-Level Vocabulary Engine*

### *Review*

## 3rd Iteration ##

Yes, you have diagnosed the problem perfectly. The primary culprit behind the the a a the saha the... is your extremely small dataset size (only 3 to 6 records).In machine learning, this is called an underfitting/data-starvation deadlock. A Transformer model is essentially a blank calculator of word patterns. Because your dataset only has a few short sentences, the model doesn't have enough reference material to learn the rules of grammar, sentence structure, or how to connect a prompt to a logical answer. Instead, it gets stuck in a loop repeating the most common filler words (the, a) and names (saha) to minimize its internal error score.To get a human-like, accurate answer without importing any pre-trained AI weights, you have two options depending on your goals:Option 1: The Code Fix (How to optimize your current setup)If you want to keep training this model yourself from scratch, we must force the tiny network to memorize your dataset exactly (forcing a deliberate overfitting state). To break the repetition loop right now, we need to make the model bigger, train it longer, and change the way it reads the text.

Apply these 3 updates to your code cells:
1. Expand the Network Layers (Update Cell 6)
Make the model "deeper" so it can memorize more complex patterns. Change your MiniCausalTransformer settings to match this larger layout configuration:
2. Force-Train the Model Longer (Update Cell 7)Because the dataset is small, 300 cycles aren't enough for the network weights to adjust properly. Crank the epochs up to 1000 cycles so the network aggressively minimizes the loss rate down to nearly zero:
3. Clamp the Output Selection (Update Cell 8)Your generation loop is using a high random factor (temperature = 0.4 and torch.multinomial). For a tiny dataset, randomness destroys the text structure. Force the model to be completely strict and select only the single most probable next word by updating the generation code lines:

## Option 2: 

The Data Fix (What a software company would do)If you apply the code fixes above, your model will perfectly answer the specific questions in your evaluation_set.json file because it has memorized them. However, if you ask it a slightly new question, it will fail again.

To build a model that can genuinely think and answer any question from scratch, you need to expand your dataset:

The Traditional Solution: 
You would need a file containing at least 10,000 to 100,000 pairs of questions, answers, and context snippets. You would let your laptop GPU train on that massive corpus for hours. Through thousands of records, the neural network finally learns that the word Chennai belongs near the word location or accused.